# JOINTS 2026 Main Notebook

# Import modules dan libraries

In [ ]:
import os, glob, random, re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
import sdm
from sdm import TableTensor

# Import datasets

In [2]:
_DATA_DIR_CANDIDATES = [
    "../data", "data", "../../data",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/**/", recursive=True),
]
DATA_DIR = next((d for d in _DATA_DIR_CANDIDATES if d and os.path.exists(os.path.join(d, "test_history.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("test_history.csv not found in any candidate DATA_DIR - set it manually.")
print("DATA_DIR =", os.path.abspath(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date_show"])
test_hist = pd.read_csv(f"{DATA_DIR}/test_history.csv", parse_dates=["date_show"])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["date_show"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA_DIR}/ticket_prices.csv")

DATA_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\data


### External datasets

In [3]:
_EXTERNAL_DIR_CANDIDATES = ["../external", "external", "../../external", *glob.glob("/kaggle/input/*")]
EXTERNAL_DIR = next((d for d in _EXTERNAL_DIR_CANDIDATES if os.path.exists(os.path.join(d, "cuti_bersama.csv"))), None)
if EXTERNAL_DIR is None:
    raise FileNotFoundError("cuti_bersama.csv not found in any candidate EXTERNAL_DIR - set it manually.")
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

cuti = pd.read_csv(f"{EXTERNAL_DIR}/cuti_bersama.csv", parse_dates=["date"])
ramadan = pd.read_csv(f"{EXTERNAL_DIR}/ramadan.csv", parse_dates=["date"])
school = pd.read_csv(f"{EXTERNAL_DIR}/school_holidays.csv", parse_dates=["date"])
extra_holidays = pd.read_csv(f"{EXTERNAL_DIR}/extra_national_holidays.csv", parse_dates=["date"])

EXTERNAL_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\external


# Aggregate Dataset

### Penentuan D1

In [4]:
D1_MIN_SHARE = 0.5

cinemas_per_day = train.groupby(["movie_title", "date_show"])["cinema_ids"].nunique()
first_show = train.groupby("movie_title")["date_show"].min()
wide_release = cinemas_per_day[cinemas_per_day >= D1_MIN_SHARE * cinemas_per_day.groupby(level=0).transform("max")]
train_d1 = wide_release.reset_index().groupby("movie_title")["date_show"].min()
train_d1 = train_d1[
    (first_show[train_d1.index] > train["date_show"].min())
    & (train_d1 + pd.Timedelta(days=9) <= train["date_show"].max())
]
test_d1 = test_hist.groupby("movie_title")["date_show"].min()

print(f"film train: {len(train_d1)} | film test_history: {len(test_d1)} | film test: {test['movie_title'].nunique()}")

film train: 205 | film test_history: 163 | film test: 160


### Pasangan film-bioskop

In [5]:
DAILY_COLS = {"total_ticket": "t", "occupation_rate": "occ", "total_show": "show"}

def build_window(hist, d1_map, n_days):
    df = hist[hist["movie_title"].isin(d1_map.index)].copy()
    df["day"] = (df["date_show"] - df["movie_title"].map(d1_map)).dt.days + 1
    df = df[df["day"].between(1, n_days)]
    parts = []
    for col, prefix in DAILY_COLS.items():
        p = df.pivot_table(index=["movie_title", "cinema_ids"], columns="day", values=col, fill_value=0)
        p = p.reindex(columns=range(1, n_days + 1), fill_value=0)
        p.columns = [f"{prefix}{d}" for d in p.columns]
        parts.append(p)
    out = pd.concat(parts, axis=1).reset_index()
    out["d1_date"] = out["movie_title"].map(d1_map)
    return out

train_window = build_window(train, train_d1, 10)
test_window = build_window(test_hist, test_d1, 3)

train_pairs = train_window[train_window["t3"] > 0].reset_index(drop=True)
test_pairs = test_window.merge(test[["movie_title", "cinema_ids"]].drop_duplicates(), on=["movie_title", "cinema_ids"])
print(f"pasangan train: {len(train_pairs)} | pasangan test: {len(test_pairs)}")

pasangan train: 8007 | pasangan test: 10373


### Format harian

In [6]:
HORIZONS = range(4, 11)
INPUT_COLS = [f"{p}{d}" for p in DAILY_COLS.values() for d in (1, 2, 3)]

def to_long(pairs, with_target):
    rows = []
    for h in HORIZONS:
        r = pairs[["movie_title", "cinema_ids", "d1_date"] + INPUT_COLS].copy()
        r["horizon"] = h
        r["date_show"] = r["d1_date"] + pd.Timedelta(days=h - 1)
        r["total_ticket"] = pairs[f"t{h}"].values if with_target else np.nan
        rows.append(r)
    return pd.concat(rows, ignore_index=True)

train_long = to_long(train_pairs, with_target=True).assign(is_train=1)
test_long = to_long(test_pairs, with_target=False).assign(is_train=0).merge(
    test[["id", "movie_title", "cinema_ids", "date_show"]], on=["movie_title", "cinema_ids", "date_show"], how="left"
)
full = pd.concat([train_long, test_long], ignore_index=True)
print(f"baris train: {len(train_long)} | baris test: {len(test_long)} | baris test.csv: {len(test)}")

baris train: 56049 | baris test: 72611 | baris test.csv: 72611


### Penggabungan semua sumber data

In [7]:
FORMAT_PATTERN = r"\s*\((?:IMAX )?[23]D\)\s*$"
CINEMA_CITY = pd.concat([train, test_hist]).drop_duplicates("cinema_ids").set_index("cinema_ids")["city_name"]
PRICE_DAY = {"weekday": "Weekday", "friday": "Friday", "weekend": "Weekend"}

def join_sources(df):
    n = len(df)
    df = df.copy()
    df["city_name"] = df["cinema_ids"].map(CINEMA_CITY)
    df = df.merge(holidays.rename(columns={"date": "date_show"}), on="date_show", how="left")
    df["price_day"] = df["day_tipe"].map(PRICE_DAY)
    df = df.merge(prices.rename(columns={"ceil": "price"}), on=["city_name", "price_day"], how="left")
    df["base_title"] = df["movie_title"].str.replace(FORMAT_PATTERN, "", regex=True)
    df = df.merge(movies.rename(columns={"original_title": "base_title"}), on="base_title", how="left")
    assert len(df) == n
    return df

full = join_sources(full)
print(f"full: {full.shape}")
full.isna().sum()[lambda s: s > 0]

full: (128660, 30)


total_ticket     72611
id               56049
holiday_name    122732
dtype: int64

### External datasets join

In [8]:
def join_external(df):
    n = len(df)
    df = df.merge(cuti[["date", "name"]].rename(columns={"date": "date_show", "name": "cuti_bersama_name"}), on="date_show", how="left")
    df = df.merge(ramadan[["date", "ramadan_day"]].rename(columns={"date": "date_show"}), on="date_show", how="left")
    df = df.merge(school[["date", "period"]].rename(columns={"date": "date_show", "period": "school_break"}), on="date_show", how="left")
    assert len(df) == n
    return df

full = join_external(full)
print(f"full: {full.shape}")
full.groupby("is_train")[["cuti_bersama_name", "ramadan_day", "school_break"]].count()

full: (128660, 33)


,cuti_bersama_name,ramadan_day,school_break
is_train,,,
0,2392,12729,12540
1,1141,0,6208


In [25]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,occ1,occ2,occ3,show1,...,active_mean,comp_n_cin,comp_size_cin,comp_fresh_cin,comp_n_nat,comp_size_nat,date_idx,own_idx,date_idx_rel,age_offset
0,28 YEARS LATER,015555fbcd8faef6b55a854605df0a87,2025-06-18,261.0,210.0,137.0,13.56,14.30,12.25,18.0,...,1.0,3,2.373528,2.373528,7.0,2.419718,-0.257161,0.080744,-0.337905,0
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,2025-06-18,1376.0,1033.0,946.0,39.38,23.87,22.75,23.0,...,1.0,5,1.443846,1.443846,7.0,2.419718,-0.257161,0.080744,-0.337905,0
2,28 YEARS LATER,04a2de432a8901363a05137a8e8de1d2,2025-06-18,0.0,83.0,85.0,0.00,16.10,16.14,0.0,...,1.5,1,2.125251,2.125251,7.0,2.419718,-0.257161,0.080744,-0.337905,0
3,28 YEARS LATER,073b764928864ed8fa0b6b115bfdd685,2025-06-18,159.0,55.0,43.0,19.57,38.15,43.26,9.0,...,1.0,1,2.987724,2.987724,7.0,2.419718,-0.257161,0.080744,-0.337905,0
4,28 YEARS LATER,108f9a4d181f12fd1b7e3c9adae38478,2025-06-18,643.0,458.0,447.0,12.21,14.42,15.14,30.0,...,1.0,5,2.229709,2.229709,7.0,2.419718,-0.257161,0.080744,-0.337905,0


# Preprocessing

### Drop incomplete days

In [10]:
rows_per_day = train.groupby("date_show").size().reindex(
    pd.date_range(train["date_show"].min(), train["date_show"].max()), fill_value=0
)
row_share = rows_per_day / rows_per_day.rolling(15, center=True, min_periods=5).median()
GAP_DAYS = row_share.index[row_share < 0.7]

def drop_gap_rows(df):
    d13_on_gap = np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(GAP_DAYS) for k in range(3)], axis=0)
    broken = (df["is_train"] == 1) & (df["date_show"].isin(GAP_DAYS) | d13_on_gap)
    return df[~broken].reset_index(drop=True)

print("gap days:", [d.date().isoformat() for d in GAP_DAYS])
n_before = len(full)
full = drop_gap_rows(full)
print(f"dropped train rows: {n_before - len(full)} | full: {full.shape}")

gap days: ['2025-06-07', '2025-06-09', '2025-06-11', '2025-06-13', '2025-06-14', '2025-06-15', '2025-06-16']
dropped train rows: 1378 | full: (127282, 33)


### Calendar flags

In [11]:
def add_calendar_flags(df):
    df = df.copy()
    df["is_holiday"] = (df["holiday_tipe"] == "holiday").astype(int)
    df["is_cuti_bersama"] = df["cuti_bersama_name"].notna().astype(int)
    df["is_school_holiday"] = df["school_break"].notna().astype(int)
    return df

full = add_calendar_flags(full)
full.groupby("is_train")[["is_holiday", "is_cuti_bersama", "is_school_holiday"]].mean().round(3)

,is_holiday,is_cuti_bersama,is_school_holiday
is_train,,,
0,0.041,0.033,0.173
1,0.054,0.018,0.114


### Genre columns

In [12]:
def add_genres(df, genre_cols=None):
    dummies = df["genre"].str.get_dummies(sep=", ").add_prefix("genre_")
    dummies.columns = dummies.columns.str.replace(" ", "_")
    if genre_cols is not None:
        dummies = dummies.reindex(columns=genre_cols, fill_value=0)
    return pd.concat([df, dummies], axis=1), list(dummies.columns)

full, GENRE_COLS = add_genres(full)
print(f"genres: {len(GENRE_COLS)}")
full.loc[full["is_train"] == 1, GENRE_COLS].mean().sort_values(ascending=False).round(3).head(10)

genres: 27


genre_Horror       0.375
genre_Drama        0.328
genre_Action       0.212
genre_Thriller     0.146
genre_Comedy       0.120
genre_Animation    0.070
genre_Mystery      0.063
genre_Fantasy      0.056
genre_Superhero    0.052
genre_Family       0.043
dtype: float64

### Category columns

In [13]:
CAT_COLS = ["city_name", "day_tipe", "age_rating"]
CAT_LEVELS = {col: full[col].astype("category").cat.categories for col in CAT_COLS}

def set_categories(df):
    df = df.copy()
    for col in CAT_COLS:
        df[col] = pd.Categorical(df[col], categories=CAT_LEVELS[col])
    return df

full = set_categories(full)
full[CAT_COLS].nunique()

city_name     69
day_tipe       3
age_rating     4
dtype: int64

### Drop columns without signal

In [14]:
DROP_COLS = [
    "holiday_tipe", "holiday_name", "cuti_bersama_name", "school_break", "genre",
    "price_day", "base_title", "producer", "director", "writer", "casts",
]

def drop_unused(df):
    return df.drop(columns=DROP_COLS)

full = drop_unused(full)
print(f"full: {full.shape}")
print(list(full.columns))

full: (127282, 52)
['movie_title', 'cinema_ids', 'd1_date', 't1', 't2', 't3', 'occ1', 'occ2', 'occ3', 'show1', 'show2', 'show3', 'horizon', 'date_show', 'total_ticket', 'is_train', 'id', 'city_name', 'day_tipe', 'price', 'age_rating', 'ramadan_day', 'is_holiday', 'is_cuti_bersama', 'is_school_holiday', 'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Biography', 'genre_Comedy', 'genre_Crime', 'genre_Disaster', 'genre_Documentary', 'genre_Drama', 'genre_Family', 'genre_Fantasy', 'genre_Gothic', 'genre_History', 'genre_Horror', 'genre_Music', 'genre_Musical', 'genre_Mystery', 'genre_Religi', 'genre_Romance', 'genre_Sci-Fi', 'genre_Sport', 'genre_Superhero', 'genre_Survival', 'genre_Suspense', 'genre_Teen', 'genre_Thriller', 'genre_War']


### Target

In [15]:
def add_target(df):
    df = df.copy()
    df["scale"] = df[["t1", "t2", "t3"]].mean(axis=1).clip(lower=1)
    df["target_ratio"] = df["total_ticket"] / df["scale"]
    return df

full = add_target(full)
full.loc[full["is_train"] == 1].groupby("horizon")["target_ratio"].median().round(3)

horizon
4     0.915
5     0.616
6     0.467
7     0.360
8     0.163
9     0.000
10    0.000
Name: target_ratio, dtype: float64

# Feature Engineering

### Days-off blocks

In [16]:
cal = pd.DataFrame({"date": pd.date_range(full["d1_date"].min() - pd.Timedelta(days=30), full["date_show"].max())})
NATIONAL_HOLIDAYS = pd.concat([holidays.loc[holidays["holiday_tipe"] == "holiday", "date"], extra_holidays["date"]])
cal["off"] = (
    (cal["date"].dt.dayofweek >= 5) | cal["date"].isin(NATIONAL_HOLIDAYS) | cal["date"].isin(cuti["date"])
).astype(int)
block_id = (cal["off"] != cal["off"].shift()).cumsum()
cal["block_len"] = cal.groupby(block_id)["off"].transform("size") * cal["off"]
BLOCK_LEN = cal.set_index("date")["block_len"]
OFF_DAY = cal.set_index("date")["off"]

def add_off_block(df):
    return df.assign(off_block_len=df["date_show"].map(BLOCK_LEN))

full = add_off_block(full)

print("longest blocks:")
print(cal[cal["block_len"] >= 4].groupby(block_id)["date"].agg(["min", "max", "size"]).reset_index(drop=True))
full.groupby("is_train")["off_block_len"].value_counts(normalize=True).unstack().round(3)

longest blocks:
         min        max  size
0 2025-03-28 2025-04-07    11
1 2025-05-10 2025-05-13     4
2 2025-05-29 2025-06-01     4
3 2025-06-06 2025-06-09     4
4 2025-12-25 2025-12-28     4
5 2026-02-14 2026-02-17     4
6 2026-03-18 2026-03-24     7


off_block_len,0,1,2,3,4,7,11
is_train,,,,,,,
0,0.658,0.005,0.228,0.019,0.047,0.042,NaN
1,0.657,0.006,0.213,0.072,0.051,NaN,0.001


### Pair features

In [17]:
D13 = ["t1", "t2", "t3"]

def add_pair_features(df):
    df = df.copy()
    df["trend"] = (df["t3"] + 1) / (df["t1"] + 1)
    df["t3_share"] = df["t3"] / df[D13].sum(axis=1)
    df["occ_mean"] = df[["occ1", "occ2", "occ3"]].mean(axis=1)
    df["show_mean"] = df[["show1", "show2", "show3"]].mean(axis=1)
    df["tickets_per_show3"] = df["t3"] / df["show3"].replace(0, np.nan)
    df["active_days"] = (df[D13] > 0).sum(axis=1)
    return df

full = add_pair_features(full)
full[["trend", "t3_share", "occ_mean", "show_mean", "tickets_per_show3", "active_days"]].describe().round(2)

,trend,t3_share,occ_mean,show_mean,tickets_per_show3,active_days
count,127282.00,127282.00,127282.00,127282.00,127282.00,127282.00
mean,11.13,0.33,19.76,8.34,32.42,2.90
std,51.19,0.16,16.83,11.10,31.17,0.36
min,0.01,0.00,0.00,0.33,0.25,1.00
25%,0.56,0.24,7.87,3.33,12.14,3.00
50%,0.89,0.32,14.80,5.00,24.00,3.00
75%,1.49,0.40,26.28,9.00,42.20,3.00
max,1199.00,1.00,100.00,216.00,421.33,3.00


### Film features

In [18]:
def film_features(window):
    g = window.groupby("movie_title")
    out = g[D13].sum().add_prefix("mv_")
    out["mv_clusters"] = g["t3"].apply(lambda s: (s > 0).sum())
    out["mv_trend"] = (out["mv_t3"] + 1) / (out["mv_t1"] + 1)
    return out

FILM_COLS = ["mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend"]

def add_film_features(df, window):
    df = df.copy()
    feats = film_features(window)
    for col in FILM_COLS:
        df[col] = df["movie_title"].map(feats[col]).values
    df["pair_share"] = df[D13].sum(axis=1) / df[["mv_t1", "mv_t2", "mv_t3"]].sum(axis=1)
    return df

is_tr = full["is_train"] == 1
full = pd.concat([add_film_features(full[is_tr], train_window), add_film_features(full[~is_tr], test_window)]).sort_index()
full.groupby("is_train")[FILM_COLS + ["pair_share"]].median().round(3)

,mv_t1,mv_t2,mv_t3,mv_clusters,mv_trend,pair_share
is_train,,,,,,
0,11615.0,12300.0,12249.0,87.0,0.869,0.008
1,22042.0,18511.0,17704.0,81.0,0.923,0.010


### Cinema features

In [19]:
all_windows = pd.concat([train_window, test_window], ignore_index=True)
cinema_feats = (
    all_windows.assign(pair_scale=all_windows[D13].mean(axis=1))
    .groupby("cinema_ids")
    .agg(cl_scale=("pair_scale", "mean"), cl_movies=("movie_title", "nunique"))
)

def add_cinema_features(df):
    return df.join(cinema_feats, on="cinema_ids")

full = add_cinema_features(full)
cinema_feats.describe().round(2)

,cl_scale,cl_movies
count,121.00,121.00
mean,192.95,192.13
std,165.41,73.64
min,18.77,5.00
25%,106.63,145.00
50%,160.53,189.00
75%,221.98,239.00
max,1348.26,347.00


### Weekday and price features

In [20]:
DAY_TIPE_BY_DATE = holidays.set_index("date")["day_tipe"]
PRICE_MAP = prices.set_index(["city_name", "price_day"])["ceil"]

def add_weekday_price(df):
    df = df.copy()
    df["dow"] = df["date_show"].dt.dayofweek
    df["d1_dow"] = df["d1_date"].dt.dayofweek

    def price_on(dates):
        keys = pd.MultiIndex.from_arrays([df["city_name"].astype(str), dates.map(DAY_TIPE_BY_DATE).map(PRICE_DAY)])
        return pd.Series(PRICE_MAP.reindex(keys).values, index=df.index)

    d13_price = pd.concat([price_on(df["d1_date"] + pd.Timedelta(days=k)) for k in range(3)], axis=1).mean(axis=1)
    df["price_ratio"] = df["price"] / d13_price
    return df

full = add_weekday_price(full)
full[["dow", "d1_dow", "price_ratio"]].describe().round(3)

,dow,d1_dow,price_ratio
count,127282.0,127282.000,127282.000
mean,3.0,2.772,0.977
std,2.0,0.726,0.144
min,0.0,0.000,0.615
25%,1.0,2.000,0.875
50%,3.0,3.000,0.955
75%,5.0,3.000,1.105
max,6.0,6.000,1.500


### Days off in the D1-D3 window

In [21]:
def add_window_offdays(df):
    df = df.copy()
    df["win_off"] = sum((df["d1_date"] + pd.Timedelta(days=k)).map(OFF_DAY) for k in range(3))
    df["tgt_off"] = df["date_show"].map(OFF_DAY)
    df["off_rel"] = df["tgt_off"] - df["win_off"] / 3
    return df

full = add_window_offdays(full)
full.groupby("is_train")[["win_off", "tgt_off", "off_rel"]].mean().round(3)

,win_off,tgt_off,off_rel
is_train,,,
0,1.059,0.342,-0.011
1,1.098,0.343,-0.023


### Late-start pairs

In [22]:
def add_late_start(df):
    df = df.copy()
    t = df[D13].values
    first = np.select([t[:, 0] > 0, t[:, 1] > 0, t[:, 2] > 0], [1, 2, 3], 4)
    last = np.select([t[:, 2] > 0, t[:, 1] > 0], [t[:, 2], t[:, 1]], t[:, 0])
    df["first_active"] = first
    df["eff_age"] = df["horizon"] - first + 1
    df["last_ratio"] = last / df["scale"]
    df["active_mean"] = t.sum(axis=1) / np.clip((t > 0).sum(axis=1), 1, None) / df["scale"]
    return df

full = add_late_start(full)
full.groupby("is_train")["first_active"].value_counts(normalize=True).unstack().round(3)

first_active,1,2,3
is_train,,,
0,0.916,0.066,0.018
1,0.928,0.055,0.018


### Competition from other films' openings

In [23]:
COMP_MIN_SIZE = 50   # film D1-D3 mean below this is too small for the date strength

def opening_pairs(df):
    return df.drop_duplicates(["movie_title", "cinema_ids"])[["movie_title", "cinema_ids", "d1_date", "t1", "t2", "t3", "scale"]]

def film_openings(df):
    f = opening_pairs(df).groupby(["movie_title", "d1_date"])[D13].sum().reset_index()
    f["m"] = f[D13].mean(axis=1)
    return f[f["m"] >= COMP_MIN_SIZE].reset_index(drop=True)

def typical_opening_shape(df):
    """usual film D_k / mean(D1-D3), k = 1..3, by release weekday"""
    f = film_openings(df)
    log_r = lambda g, k: np.average(np.log((g[f"t{k}"] + 1) / (g["m"] + 1)), weights=np.sqrt(g["m"]))
    overall = [np.exp(log_r(f, k)) for k in (1, 2, 3)]
    by_dow = {dw: [np.exp(log_r(g, k)) for k in (1, 2, 3)] for dw, g in f.groupby(f["d1_date"].dt.dayofweek) if len(g) >= 5}
    return {dw: by_dow.get(dw, overall) for dw in range(7)}

def opening_surprise(df, shape):
    """log of each film's D1-D3 days vs the usual shape for its release weekday, one row per film and date"""
    rows = []
    for r in film_openings(df).itertuples():
        t = shape[r.d1_date.dayofweek]
        for k in range(3):
            rows.append((r.movie_title, r.d1_date + pd.Timedelta(days=k), np.log((getattr(r, f"t{k + 1}") + 1) / (r.m + 1) / t[k]), np.sqrt(r.m)))
    s = pd.DataFrame(rows, columns=["movie_title", "date", "s", "w"])
    return s.assign(sw=s["s"] * s["w"])

def date_strength(dates, titles, surprise):
    """size-weighted mean surprise of the other films in their D1-D3 on each date (NaN when there are none)"""
    tot = surprise.groupby("date")[["sw", "w"]].sum().reindex(pd.DatetimeIndex(dates)).values
    own = surprise.set_index(["date", "movie_title"])[["sw", "w"]].reindex(pd.MultiIndex.from_arrays([pd.DatetimeIndex(dates), titles])).fillna(0).values
    w = tot[:, 1] - own[:, 1]
    ok = w > 1e-9
    return np.where(ok, (tot[:, 0] - own[:, 0]) / np.where(ok, w, 1), np.nan)

def add_competition(df, ref, shape):
    """ref: rows whose pairs are the other films' openings"""
    df = df.reset_index(drop=True)
    others = opening_pairs(ref)
    own_size = opening_pairs(df).groupby("movie_title")["scale"].sum()

    key = df[["movie_title", "cinema_ids", "d1_date", "date_show"]].reset_index()
    m = key.merge(others[["cinema_ids", "movie_title", "d1_date", "scale"]], on="cinema_ids", suffixes=("", "_o"))
    m = m[(m["movie_title_o"] != m["movie_title"]) & (m["d1_date_o"] > m["d1_date"]) & (m["d1_date_o"] <= m["date_show"])]
    fresh = m[(m["date_show"] - m["d1_date_o"]).dt.days <= 2].groupby("index")["scale"].sum()
    df["comp_n_cin"] = m.groupby("index").size().reindex(df.index, fill_value=0).values
    df["comp_size_cin"] = np.log1p(m.groupby("index")["scale"].sum().reindex(df.index, fill_value=0).values / df["scale"].values)
    df["comp_fresh_cin"] = np.log1p(fresh.reindex(df.index, fill_value=0).values / df["scale"].values)

    films = others.groupby("movie_title").agg(fd1=("d1_date", "min"), fsize=("scale", "sum")).rename_axis("movie_o").reset_index()
    u = df[["movie_title", "d1_date", "date_show"]].drop_duplicates()
    mm = u.merge(films, how="cross")
    mm = mm[(mm["movie_o"] != mm["movie_title"]) & (mm["fd1"] > mm["d1_date"]) & (mm["fd1"] <= mm["date_show"])]
    u = u.merge(mm.groupby(["movie_title", "d1_date", "date_show"]).agg(comp_n_nat=("movie_o", "size"), comp_size_nat=("fsize", "sum")).reset_index(),
                how="left", on=["movie_title", "d1_date", "date_show"])

    surprise = opening_surprise(ref, shape)
    u["date_idx"] = date_strength(u["date_show"], u["movie_title"].values, surprise)
    own_days = np.vstack([date_strength(u["d1_date"] + pd.Timedelta(days=k), u["movie_title"].values, surprise) for k in range(3)])
    n_days = (~np.isnan(own_days)).sum(axis=0)
    u["own_idx"] = np.where(n_days > 0, np.nansum(own_days, axis=0) / np.maximum(n_days, 1), np.nan)
    u["date_idx_rel"] = u["date_idx"] - u["own_idx"]

    df = df.merge(u, how="left", on=["movie_title", "d1_date", "date_show"])
    df["comp_n_nat"] = df["comp_n_nat"].fillna(0)
    df["comp_size_nat"] = np.log1p(df["comp_size_nat"].fillna(0) / df["movie_title"].map(own_size))
    return df

COMP_COLS = ["comp_n_cin", "comp_size_cin", "comp_fresh_cin", "comp_n_nat", "comp_size_nat", "date_idx", "own_idx", "date_idx_rel"]
OPENING_SHAPE = typical_opening_shape(full[full["is_train"] == 1])
is_tr = full["is_train"] == 1
full = pd.concat([add_competition(full[is_tr], full[is_tr], OPENING_SHAPE),
                  add_competition(full[~is_tr], full[~is_tr], OPENING_SHAPE)], ignore_index=True)
full.groupby("is_train")[COMP_COLS].mean().round(3)

,comp_n_cin,comp_size_cin,comp_fresh_cin,comp_n_nat,comp_size_nat,date_idx,own_idx,date_idx_rel
is_train,,,,,,,,
0,2.645,1.301,0.898,4.475,1.438,0.005,-0.012,0.020
1,2.430,1.256,0.874,5.446,1.485,-0.000,0.004,-0.003


### Extra training rows (shifted windows)

In [24]:
AUG_SHIFTS = [1, 2, 3, 5, 7, 10, 14]

def build_shifted_rows(shift):
    d1 = train_d1 + pd.Timedelta(days=shift)
    d1 = d1[d1 + pd.Timedelta(days=9) <= train["date_show"].max()]
    window = build_window(train, d1, 10)
    df = to_long(window[window["t3"] > 0].reset_index(drop=True), with_target=True).assign(is_train=1)
    df = drop_gap_rows(join_external(join_sources(df)))
    df, _ = add_genres(add_calendar_flags(df), GENRE_COLS)
    df = add_target(drop_unused(set_categories(df)))
    df = add_weekday_price(add_cinema_features(add_film_features(add_pair_features(add_off_block(df)), window)))
    df = add_late_start(add_window_offdays(df))
    df = add_competition(df, full[full["is_train"] == 1], OPENING_SHAPE)
    return df.assign(age_offset=shift)

full["age_offset"] = 0
AUG = pd.concat([build_shifted_rows(s) for s in AUG_SHIFTS], ignore_index=True)
print(f"extra rows: {len(AUG)} ({len(AUG) / (full['is_train'] == 1).sum():.1f}x the real train rows)")
AUG.groupby("age_offset")["target_ratio"].agg(["size", "mean"]).round(3)

extra rows: 254483 (4.7x the real train rows)


,size,mean
age_offset,,
1,50854,0.464
2,50833,0.482
3,48573,0.521
5,33873,0.635
7,28516,0.591
10,26786,0.410
14,15048,0.591


In [25]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,occ1,occ2,occ3,show1,...,active_mean,comp_n_cin,comp_size_cin,comp_fresh_cin,comp_n_nat,comp_size_nat,date_idx,own_idx,date_idx_rel,age_offset
0,28 YEARS LATER,015555fbcd8faef6b55a854605df0a87,2025-06-18,261.0,210.0,137.0,13.56,14.30,12.25,18.0,...,1.0,3,2.373528,2.373528,7.0,2.419718,-0.257161,0.080744,-0.337905,0
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,2025-06-18,1376.0,1033.0,946.0,39.38,23.87,22.75,23.0,...,1.0,5,1.443846,1.443846,7.0,2.419718,-0.257161,0.080744,-0.337905,0
2,28 YEARS LATER,04a2de432a8901363a05137a8e8de1d2,2025-06-18,0.0,83.0,85.0,0.00,16.10,16.14,0.0,...,1.5,1,2.125251,2.125251,7.0,2.419718,-0.257161,0.080744,-0.337905,0
3,28 YEARS LATER,073b764928864ed8fa0b6b115bfdd685,2025-06-18,159.0,55.0,43.0,19.57,38.15,43.26,9.0,...,1.0,1,2.987724,2.987724,7.0,2.419718,-0.257161,0.080744,-0.337905,0
4,28 YEARS LATER,108f9a4d181f12fd1b7e3c9adae38478,2025-06-18,643.0,458.0,447.0,12.21,14.42,15.14,30.0,...,1.0,5,2.229709,2.229709,7.0,2.419718,-0.257161,0.080744,-0.337905,0


# Regressor Models

### Setup and validation

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

RUN_MODELS = ["lgbm", "kumo"]   # ["lgbm"] alone is much faster
MODEL_SEEDS = [42, 7, 123]
ZERO_SNAP = {"lgbm": 0.2}
N_FOLDS = 5
USE_EXTRA_ROWS = True   # shifted-window rows of the fitting films are added in every fold (validation stays on real rows)
EXTRA_WEIGHT = 0.5      # sample weight of the shifted-window rows in the tree models

NUM_FEATURES = [
    "t1", "t2", "t3", "occ1", "occ2", "occ3", "show1", "show2", "show3", "horizon", "scale", "price",
    "trend", "t3_share", "occ_mean", "show_mean", "tickets_per_show3", "active_days",
    "mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend", "pair_share", "cl_scale", "cl_movies",
    "dow", "d1_dow", "price_ratio",
    "is_holiday", "is_cuti_bersama", "is_school_holiday", "off_block_len",
    "win_off", "tgt_off", "off_rel",
    "first_active", "eff_age", "last_ratio", "active_mean",
] + COMP_COLS + ["age_offset"] + GENRE_COLS
FEATURES = NUM_FEATURES + CAT_COLS

train_df = full[full["is_train"] == 1].reset_index(drop=True)
test_df = full[full["is_train"] == 0].reset_index(drop=True)
X, y, groups = train_df[FEATURES], train_df["target_ratio"].values, train_df["movie_title"]
X_test = test_df[FEATURES]
X_aug, y_aug = AUG[FEATURES], AUG["target_ratio"].values
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups))

OOF, TEST_PRED, SCORES = {}, {}, {}

def mase_ratio(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def fit_rows(idx):
    """real rows idx + the shifted-window rows of the same films; is_extra marks the shifted rows"""
    X_fit, y_fit = X.iloc[idx], y[idx]
    if not USE_EXTRA_ROWS:
        return X_fit, y_fit, np.zeros(len(idx), bool)
    keep = AUG["movie_title"].isin(set(groups.iloc[idx])).values
    return (pd.concat([X_fit, X_aug[keep]], ignore_index=True), np.r_[y_fit, y_aug[keep]],
            np.r_[np.zeros(len(idx), bool), np.ones(keep.sum(), bool)])

def extra_weights(is_extra):
    return np.where(is_extra, EXTRA_WEIGHT, 1.0)

def seed_average(make_model, X_tr, y_tr, X_pred, prep=lambda d: d, sample_weight=None):
    preds = [make_model(seed).fit(prep(X_tr), y_tr, sample_weight=sample_weight).predict(prep(X_pred)) for seed in MODEL_SEEDS]
    return np.mean(preds, axis=0)

def cross_validate(name, fit_predict):
    snap = lambda p: np.where(p < ZERO_SNAP.get(name, 0.0), 0.0, p)
    oof = np.zeros(len(train_df))
    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        oof[va_idx] = snap(np.clip(fit_predict(*fit_rows(tr_idx), X.iloc[va_idx]), 0, None))
        fold_scores.append(mase_ratio(y[va_idx], oof[va_idx]))
        print(f"{name} fold {fold}: MASE={fold_scores[-1]:.4f}")
    OOF[name], SCORES[name] = oof, fold_scores
    print(f"{name} CV MASE: {mase_ratio(y, oof):.4f}")
    TEST_PRED[name] = snap(np.clip(fit_predict(*fit_rows(np.arange(len(X))), X_test), 0, None))

print(f"features: {len(FEATURES)} | train rows: {len(train_df)} | extra rows: {len(AUG) if USE_EXTRA_ROWS else 0} | test rows: {len(test_df)}")
print(f"naive predict 0       : {mase_ratio(y, 0):.4f}")
print(f"naive predict D1-D3 avg: {mase_ratio(y, 1):.4f}")

### LGBM Regressor

In [27]:
LGBM_PARAMS = dict(
    n_estimators=1500,   # 883 in the Optuna run; 1500 is better with the extra rows
    learning_rate=0.0149601,
    num_leaves=87,
    min_child_samples=11,
    subsample=0.641476,
    colsample_bytree=0.48309,
    reg_lambda=1.85904,
    reg_alpha=11.5708,
    subsample_freq=1,
)

def lgbm_fit_predict(X_tr, y_tr, is_extra, X_pred):
    make_model = lambda seed: LGBMRegressor(objective="l1", random_state=seed, verbose=-1, **LGBM_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred, sample_weight=extra_weights(is_extra))

if "lgbm" in RUN_MODELS:
    cross_validate("lgbm", lgbm_fit_predict)

lgbm fold 0: MASE=0.2695
lgbm fold 1: MASE=0.4908
lgbm fold 2: MASE=0.2433
lgbm fold 3: MASE=0.3730
lgbm fold 4: MASE=0.2736
lgbm CV MASE: 0.3300


### Kumo-Tabular

In [30]:
KUMO_CONTEXT_ROWS = 8000   # real rows in the context
KUMO_EXTRA_ROWS = 2000     # shifted-window rows in the context (8k + 2k had the best CV of 10k + 0, 8k + 2k, 6k + 4k, 4k + 6k)
KUMO_CONTEXT_SEEDS = [42, 7, 123]
KUMO_STYPES = {c: "numerical" for c in NUM_FEATURES} | {c: "categorical" for c in CAT_COLS}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
kumo_model = sdm.models.KumoTabular(task="regression", size="large", device=DEVICE).eval() if "kumo" in RUN_MODELS else None

def kumo_fit_predict(X_tr, y_tr, is_extra, X_pred, chunk_rows=2000):
    real_idx, extra_idx = np.flatnonzero(~is_extra), np.flatnonzero(is_extra)
    preds = []
    for seed in KUMO_CONTEXT_SEEDS:
        rng = np.random.default_rng(seed)
        ctx = np.r_[rng.choice(real_idx, min(KUMO_CONTEXT_ROWS, len(real_idx)), replace=False),
                    rng.choice(extra_idx, min(KUMO_EXTRA_ROWS, len(extra_idx)), replace=False)].astype(int)
        x_ctx = TableTensor.from_pandas(X_tr.iloc[ctx], KUMO_STYPES, device=DEVICE)
        y_ctx = TableTensor.from_pandas(pd.DataFrame({"y": y_tr[ctx]}), {"y": "numerical"}, device=DEVICE)
        kumo_model.fit(x_ctx, y_ctx, num_estimators=4, generator=torch.Generator(device=DEVICE).manual_seed(seed))
        quantiles = []
        for start in range(0, len(X_pred), chunk_rows):
            x_q = TableTensor.from_pandas(X_pred.iloc[start:start + chunk_rows], KUMO_STYPES, device=DEVICE)
            quantiles.append(kumo_model.predict(x_q).numerical.float().cpu())
        q = torch.cat(quantiles).numpy()
        preds.append(q[:, q.shape[1] // 2])
        torch.cuda.empty_cache()
    return np.mean(preds, axis=0)

if "kumo" in RUN_MODELS:
    cross_validate("kumo", kumo_fit_predict)

C:\Users\leouw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdm\tensor\string.py:502: RuntimeWarning: Falling back to a CPU-based string sort because cuDF is not installed. Install cuDF to enable faster CUDA-based string sorting without device synchronization. This warning will be suppressed for the remainder of this process.
  warn_once(
C:\Users\leouw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdm\relational\join.py:73: RuntimeWarning: Falling back to a CPU-based join because cuDF is not installed. Install cuDF to enable faster CUDA-based joins without device synchronization. This warning will be suppressed for the remainder of this process.
  warn_once(


kumo fold 0: MASE=0.2636
kumo fold 1: MASE=0.4840
kumo fold 2: MASE=0.2409
kumo fold 3: MASE=0.3632
kumo fold 4: MASE=0.2666
kumo CV MASE: 0.3237


### Model comparison

In [31]:
comparison = pd.DataFrame(
    {name: [mase_ratio(y, OOF[name])] + SCORES[name] for name in OOF},
    index=["CV"] + [f"fold {k}" for k in range(N_FOLDS)],
).T.sort_values("CV")
comparison.round(4)

,CV,fold 0,fold 1,fold 2,fold 3,fold 4
kumo,0.3237,0.2636,0.4840,0.2409,0.3632,0.2666
lgbm,0.3300,0.2695,0.4908,0.2433,0.3730,0.2736


### Blend

In [32]:
BLEND_WEIGHTS = {"kumo": 0.7, "lgbm": 0.3}
BLEND_SNAP = 0.1
used = {m: w for m, w in BLEND_WEIGHTS.items() if m in OOF and w > 0}
total = sum(used.values())
snap_blend = lambda p: np.where(p < BLEND_SNAP, 0.0, p)
oof_blend = snap_blend(sum(w * OOF[m] for m, w in used.items()) / total)
test_blend = snap_blend(sum(w * TEST_PRED[m] for m, w in used.items()) / total)
print(f"blend {({m: round(w / total, 3) for m, w in used.items()})}: CV MASE {mase_ratio(y, oof_blend):.4f}")
if {"kumo", "lgbm"} <= set(OOF):
    for w in (0.5, 0.6, 0.7, 0.8):
        print(f"  kumo {w:.1f} / lgbm {1 - w:.1f}: CV {mase_ratio(y, snap_blend(w * OOF['kumo'] + (1 - w) * OOF['lgbm'])):.4f}")
train_df.assign(err=np.abs(y - oof_blend)).groupby("horizon")["err"].mean().round(4)

blend {'kumo': 0.7, 'lgbm': 0.3}: CV MASE 0.3226
  kumo 0.5 / lgbm 0.5: CV 0.3238
  kumo 0.6 / lgbm 0.4: CV 0.3230
  kumo 0.7 / lgbm 0.3: CV 0.3226
  kumo 0.8 / lgbm 0.2: CV 0.3226


horizon
4     0.3499
5     0.3650
6     0.3271
7     0.3210
8     0.2829
9     0.3020
10    0.3102
Name: err, dtype: float64

# Postprocessing

### Level to a reference submission (LB-tuned, no external data)

In [ ]:
test_submit = test_blend
USE_LEVEL_REF = True
REF_SUBMISSION = "submission_ref_lb0.43729.csv"   # LB-probe file, no Cinepoint data
LEVEL_WEIGHT = 0.5
RAMADAN_START, LEBARAN_START = ramadan["date"].min(), pd.Timestamp("2026-03-20")

def season_group(df):
    d13_ramadan = np.any([((df["d1_date"] + pd.Timedelta(days=k)) >= RAMADAN_START) & ((df["d1_date"] + pd.Timedelta(days=k)) < LEBARAN_START) for k in range(3)], axis=0)
    in_ramadan = (df["date_show"] >= RAMADAN_START) & (df["date_show"] < LEBARAN_START)
    xmas = (df["date_show"] >= "2025-12-20") & (df["date_show"] <= "2026-01-04")
    return np.select([d13_ramadan & ~in_ramadan, d13_ramadan, in_ramadan, xmas],
                     ["ram_to_lebaran", "ram_to_ram", "pre_to_ram", "xmas"], "normal")

if USE_LEVEL_REF:
    ref_ratio = pd.read_csv(REF_SUBMISSION).set_index("id")["total_ticket"].reindex(test_df["id"].astype(int)).values / test_df["scale"].values
    cells_df = pd.DataFrame({"group": season_group(test_df), "horizon": test_df["horizon"].values, "blend": test_blend, "ref": ref_ratio})
    by_cell = cells_df.groupby(["group", "horizon"])
    factor = by_cell["ref"].transform("mean") / by_cell["blend"].transform("mean")
    test_submit = LEVEL_WEIGHT * (cells_df["blend"] * factor).values + (1 - LEVEL_WEIGHT) * ref_ratio
    print(f"rows per group: {cells_df['group'].value_counts().to_dict()}")
    print(f"mean abs change of the ratio vs the reference: {np.abs(test_submit - ref_ratio).mean():.4f}")
    display((by_cell["ref"].mean() / by_cell["blend"].mean()).unstack().round(2))
test_df["total_ticket"] = test_submit * test_df["scale"]
test_df["total_ticket"].sum()

### Timing shape from last year's films (Cinepoint up to 2025-09-30)

In [ ]:
USE_TIMING_SHAPE = True
TIMING_ALPHA = 0.5   # strength of the shape correction (1 = full analog shape)
TIMING_FILES = ["cinepoint_daily_top_prev.csv", "cinepoint_daily_top_train.csv"]   # Cinepoint up to 2025-09-30 only
EID_DATES = pd.to_datetime(["2024-04-10", "2025-03-31", "2026-03-21"])            # Idulfitri day 1 (national holiday)
XMAS_DATES = pd.to_datetime(["2023-12-25", "2024-12-25", "2025-12-25"])
EID_RANGE, XMAS_RANGE = (-40, 14), (-14, 10)    # release days (from Eid / Christmas) that get an analog shape
HZ = list(range(4, 11))
CP_FORMAT = r"\s*\((?:IMAX[^)]*|3D|2D|4DX)\)\s*$"
cp_key = lambda s: re.sub(r"[^a-z0-9]", "", re.sub(CP_FORMAT, "", s, flags=re.I).lower())   # Cinepoint title key


def anchor_offset(d1):
    """('eid' or 'xmas', days from the nearest Eid / Christmas) when the release falls inside the ranges, else (None, None)."""
    for anchors, kind, (lo, hi) in [(EID_DATES, "eid", EID_RANGE), (XMAS_DATES, "xmas", XMAS_RANGE)]:
        k = (d1 - anchors).days
        hit = (k >= lo) & (k <= hi)
        if hit.any():
            return kind, int(k[hit][0])
    return None, None


def history_curves(min_d13_mean=3000):
    """Real national D4-D10 / D1-D3 of every pre-cutoff Cinepoint film whose D1 can be trusted."""
    cpd = pd.concat([pd.read_csv(f"{EXTERNAL_DIR}/cinepoint/{n}", parse_dates=["date"]) for n in TIMING_FILES], ignore_index=True)
    assert cpd["date"].max() <= pd.Timestamp("2025-09-30"), "post-cutoff Cinepoint data in the timing files"
    cpd = cpd.drop_duplicates(["date", "rank"]).assign(key=lambda d: d["title"].map(cp_key))
    daily = cpd.groupby(["key", "date"])["daily_admission"].sum()
    scraped = set(cpd["date"])
    rows = []
    for key, s in daily.groupby(level=0):
        s = s.droplevel(0)
        d1 = s.index.min()
        # D1 = first day in the chart, trusted only when the week before it and D1-D10 were all scraped
        if not all(d in scraped for d in pd.date_range(d1 - pd.Timedelta(days=7), periods=17)):
            continue
        a = s.reindex(pd.date_range(d1, periods=10))
        if a.iloc[:3].isna().any() or a.iloc[:3].mean() < min_d13_mean:
            continue
        rows.append({"key": key, "d1": d1, "m13": a.iloc[:3].mean(), **{f"r{h}": a.iloc[h - 1] / a.iloc[:3].mean() for h in HZ}})
    hist = pd.DataFrame(rows)
    hist[["kind", "off"]] = [anchor_offset(d) for d in hist["d1"]]
    return hist


def analog_shape(hist, kind, off, width=4, min_films=3, max_width=10):
    """Size-weighted mean D4-D10 log shape of last years' films released at the same offset (+- width days)."""
    while width <= max_width:
        a = hist[(hist["kind"] == kind) & ((hist["off"] - off).abs() <= width)].dropna(subset=[f"r{h}" for h in HZ])
        if len(a) >= min_films:
            shape = np.log(a[[f"r{h}" for h in HZ]].values)
            shape -= shape.mean(axis=1, keepdims=True)
            return np.average(shape, axis=0, weights=np.sqrt(a["m13"].values))
        width += 2
    return None


if USE_TIMING_SHAPE:
    hist = history_curves()
    # national (Cinepoint) vs dataset retention per horizon, from train films that are in both
    tr = train_df.assign(key=train_df["movie_title"].map(cp_key))
    pair_scale = tr.drop_duplicates(["movie_title", "cinema_ids"]).groupby("key")["scale"].sum()
    ds_ret = (tr["target_ratio"] * tr["scale"]).groupby([tr["key"], tr["horizon"]]).sum().unstack().div(pair_scale, axis=0)
    h_idx = hist.set_index("key")
    common = ds_ret.index.intersection(h_idx.index)
    kappa = np.array([np.nanmedian(np.log(ds_ret.loc[common, h].clip(1e-3)) - np.log(h_idx.loc[common, f"r{h}"].clip(1e-3))) for h in HZ])

    groups = season_group(test_df)
    final = test_submit
    films = test_df.drop_duplicates(["movie_title", "cinema_ids"]).groupby("movie_title").agg(d1=("d1_date", "min"), m=("scale", "sum"))
    film_group = pd.Series(groups, index=test_df.index).groupby(test_df["movie_title"]).agg(lambda s: s.mode().iloc[0])
    pred_cell = pd.Series(final * test_df["scale"].values).groupby([groups, test_df["horizon"].values]).sum()
    shape_factor = {}
    for g in ["ram_to_lebaran", "ram_to_ram", "pre_to_ram", "xmas"]:
        shapes, weights = [], []
        for f in film_group[film_group == g].index:
            kind, off = anchor_offset(films.loc[f, "d1"])
            s = analog_shape(hist, kind, off) if kind else None
            if s is not None:
                shapes.append(s); weights.append(films.loc[f, "m"])
        if not shapes or g not in pred_cell.index.get_level_values(0):
            continue
        hz = [h for h in HZ if pred_cell[g].get(h, 0) > 0]
        target = pd.Series(np.average(shapes, axis=0, weights=weights) + kappa, index=HZ)[hz]
        model = np.log(pred_cell[g][hz])
        corr = (target - target.mean()) - (model - model.mean())       # shape only: the group's level is kept
        shape_factor.update({(g, h): np.exp(TIMING_ALPHA * c) for h, c in corr.items()})
    mult = pd.Series(shape_factor).reindex(pd.MultiIndex.from_arrays([groups, test_df["horizon"].values])).fillna(1.0).values
    test_submit = test_submit * mult
    print(f"history films: {len(hist)} ({hist['d1'].min().date()} to {hist['d1'].max().date()}), with an Eid / Christmas offset: {hist['kind'].notna().sum()}")
    display(pd.Series(shape_factor).unstack().round(2))
test_df["total_ticket"] = test_submit * test_df["scale"]
test_df["total_ticket"].sum()

# Submission

In [ ]:
def write_submission(ratio, path):
    out = test_df[["id"]].astype({"id": int}).assign(total_ticket=ratio * test_df["scale"].values)
    sub = sample_sub[["id"]].merge(out, on="id", how="left")
    assert len(sub) == len(sample_sub) and sub["total_ticket"].notna().all()
    sub.to_csv(path, index=False)
    return sub

submission = write_submission(test_submit, "submission_legit.csv")
submission.describe()